In [1]:
import sys; print(sys.executable)

c:\Users\josed\.conda\envs\tiffprocessing\python.exe


# IMS to TIFF conversion

Converts Imaris `.ims` files to `.tif` so they can be used with the other notebooks in this repo.

`.ims` files are HDF5 under the hood, so this reads them directly with `h5py` (already in the `tiffprocessing` env) — no Imaris SDK or Bio-Formats/Java dependency needed.

Each channel is stored at `/DataSet/ResolutionLevel {r}/TimePoint {t}/Channel {c}/Data` as a `(Z, Y, X)` array. This notebook stacks all channels into a single `(Z, C, Y, X)` array per file and writes it as an ImageJ-hyperstack-compatible TIFF, which matches the axis order `tiff_to_h5` in `HDF5_Generation_functions.py` expects.

If your downstream pipeline needs the `hyb_###_pos##.tif` naming convention, rename the output files (or adjust `output_name_fn` in the batch cell below) before running `process_folders`.

In [2]:
import os
from pathlib import Path

import numpy as np
import h5py
import tifffile

## Inspect an IMS file (optional)

Check the resolution levels / time points / channels / shape before converting. Run this on one file to sanity-check the structure matches what the conversion functions below expect.

In [3]:
def _sorted_group_keys(group, prefix):
    """List keys like 'Channel 0', 'Channel 1', ... sorted numerically."""
    keys = [k for k in group.keys() if k.startswith(prefix)]
    return sorted(keys, key=lambda k: int(k.rsplit(" ", 1)[-1]))


def inspect_ims(ims_path):
    """Print the resolution levels, time points, channels, shape and dtype of an .ims file."""
    with h5py.File(ims_path, "r") as f:
        res_levels = _sorted_group_keys(f["DataSet"], "ResolutionLevel")
        res0 = f["DataSet"][res_levels[0]]
        time_points = _sorted_group_keys(res0, "TimePoint")
        tp0 = res0[time_points[0]]
        channels = _sorted_group_keys(tp0, "Channel")
        data0 = tp0[channels[0]]["Data"]

        print(f"File: {ims_path}")
        print(f"Resolution levels: {res_levels}")
        print(f"Time points: {time_points}")
        print(f"Channels: {channels}")
        print(f"Per-channel shape (Z, Y, X): {data0.shape}, dtype: {data0.dtype}")

        return {
            "resolution_levels": res_levels,
            "time_points": time_points,
            "channels": channels,
            "shape_zyx": data0.shape,
            "dtype": data0.dtype,
        }


# Example: point this at one of your .ims files to inspect it
# inspect_ims(r"C:\path\to\your\file.ims")

In [4]:
def get_voxel_size(ims_path, resolution_level=0):
    """
    Return the voxel size (in micrometers) of an .ims file as {'x': .., 'y': .., 'z': ..}.

    Divides the physical extents stored in DataSetInfo/Image (ExtMin/ExtMax, in um)
    by the actual voxel counts at the given resolution level, so it's correct even
    for downsampled resolution levels (not just resolution_level=0).
    """
    with h5py.File(ims_path, "r") as f:
        img_attrs = f["DataSetInfo"]["Image"].attrs

        def _attr_to_float(name):
            raw = img_attrs[name]
            chars = [c.decode() if isinstance(c, bytes) else c for c in raw]
            return float("".join(chars))

        ext_min = [_attr_to_float(f"ExtMin{i}") for i in range(3)]  # (x, y, z), in um
        ext_max = [_attr_to_float(f"ExtMax{i}") for i in range(3)]

        tp_group = f["DataSet"][f"ResolutionLevel {resolution_level}"]["TimePoint 0"]
        channel_keys = _sorted_group_keys(tp_group, "Channel")
        z, y, x = tp_group[channel_keys[0]]["Data"].shape

    voxel_size_um = {
        "x": (ext_max[0] - ext_min[0]) / x,
        "y": (ext_max[1] - ext_min[1]) / y,
        "z": (ext_max[2] - ext_min[2]) / z,
    }
    print(f"Voxel size (um): X={voxel_size_um['x']:.4f}, Y={voxel_size_um['y']:.4f}, Z={voxel_size_um['z']:.4f}")
    return voxel_size_um


# Example: point this at one of your .ims files to get its XY/Z pixel size
# get_voxel_size(r"C:\path\to\your\file.ims")

## Conversion functions

`ims_to_array` reads every channel at a given resolution level / time point and stacks them into a `(Z, C, Y, X)` numpy array. `ims_to_tiff` writes that array out as a TIFF.

In [5]:
def ims_to_array(ims_path, resolution_level=0, time_point=0):
    """
    Read an .ims file and stack all channels into a single array.

    Parameters
    ----------
    ims_path : str or Path
        Path to the input .ims file.
    resolution_level : int, optional
        Imaris resolution pyramid level (0 = full resolution).
    time_point : int, optional
        Time point index to read.

    Returns
    -------
    np.ndarray
        Array of shape (Z, C, Y, X).
    """
    with h5py.File(ims_path, "r") as f:
        tp_group = f["DataSet"][f"ResolutionLevel {resolution_level}"][f"TimePoint {time_point}"]
        channel_keys = _sorted_group_keys(tp_group, "Channel")
        if not channel_keys:
            raise ValueError(f"No channels found in {ims_path} at resolution {resolution_level}, time {time_point}.")

        channels = [tp_group[ck]["Data"][:] for ck in channel_keys]

    # Each channel is (Z, Y, X) -> stack on axis=1 to get (Z, C, Y, X)
    return np.stack(channels, axis=1)


def ims_to_tiff(ims_path, output_path=None, resolution_level=0, time_point=0):
    """
    Convert a single .ims file to a .tif file.

    Parameters
    ----------
    ims_path : str or Path
        Path to the input .ims file.
    output_path : str or Path, optional
        Path to the output .tif file, or to a folder to write it into (using the input's base name).
        Defaults to the same name/folder as the input with a .tif extension.
    resolution_level : int, optional
        Imaris resolution pyramid level (0 = full resolution).
    time_point : int, optional
        Time point index to convert.

    Returns
    -------
    str
        Path to the written .tif file.
    """
    ims_path = Path(ims_path)
    if output_path is None:
        output_path = ims_path.with_suffix(".tif")
    output_path = Path(output_path)
    if output_path.is_dir() or output_path.suffix.lower() not in (".tif", ".tiff"):
        output_path = output_path / f"{ims_path.stem}.tif"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    array = ims_to_array(ims_path, resolution_level=resolution_level, time_point=time_point)
    tifffile.imwrite(output_path, array, imagej=True, metadata={"axes": "ZCYX"})

    print(f"{ims_path.name} -> {output_path} | shape={array.shape} (Z,C,Y,X), dtype={array.dtype}")
    return str(output_path)

## Convert a single file

(Optional) Use this to test with one file first.

In [13]:
# Set paths
example_ims =r"G:\data\GFP\PS8564_6.ims"
example_tiff_output = r"G:\data\GFP\PS8564_6.tif"

# Convert IMS to TIFF
tiff_path = ims_to_tiff(example_ims, example_tiff_output)
get_voxel_size(example_ims, resolution_level=0)

PS8564_6.ims -> G:\data\GFP\PS8564_6.tif | shape=(48, 1, 2048, 2048) (Z,C,Y,X), dtype=uint16
Voxel size (um): X=0.1005, Y=0.1005, Z=0.8958


{'x': 0.10049804687499986, 'y': 0.10049804687500008, 'z': 0.8958333333333334}

## Batch convert a folder

Converts every `.ims` file directly inside `input_folder` and writes the `.tif` files to `output_folder`, keeping the original base filename.

In [ ]:
def convert_ims_folder(input_folder, output_folder, resolution_level=0, time_point=0, output_name_fn=None):
    """
    Convert every .ims file in input_folder to .tif in output_folder.

    Parameters
    ----------
    input_folder : str or Path
        Folder containing .ims files.
    output_folder : str or Path
        Folder to write .tif files to.
    resolution_level : int, optional
        Imaris resolution pyramid level (0 = full resolution).
    time_point : int, optional
        Time point index to convert.
    output_name_fn : callable, optional
        Function taking the input Path and returning the output filename (str).
        Defaults to keeping the original base name with a .tif extension.
        e.g. to rename to the hyb_###_pos## convention used by process_folders():
            output_name_fn=lambda p: f"hyb_000_{p.stem}.tif"

    Returns
    -------
    list of str
        Paths to the created .tif files. Files that fail to convert (e.g. placeholder/stub
        .ims files from a split acquisition series with no actual image data) are skipped
        with a printed warning rather than aborting the whole batch.
    """
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    ims_files = sorted(input_folder.glob("*.ims"))
    if not ims_files:
        print(f"No .ims files found in {input_folder}")
        return []

    converted = []
    skipped = []
    for ims_path in ims_files:
        out_name = output_name_fn(ims_path) if output_name_fn else f"{ims_path.stem}.tif"
        out_path = output_folder / out_name
        try:
            converted.append(
                ims_to_tiff(ims_path, out_path, resolution_level=resolution_level, time_point=time_point)
            )
        except Exception as e:
            print(f"Skipping {ims_path.name}: {type(e).__name__}: {e}")
            skipped.append(str(ims_path))

    print(f"Converted {len(converted)} file(s), skipped {len(skipped)}.")
    if skipped:
        print("Skipped files:")
        for s in skipped:
            print(f"  {s}")
    return converted


# Set folders
ims_input_folder = r"C:\Users\josed\GitHub\WormFISH\data\ims"
tiff_output_folder = r"C:\Users\josed\GitHub\WormFISH\data\tiff"

converted_files = convert_ims_folder(ims_input_folder, tiff_output_folder)